# 🚀 DeepSpaceVision — Dərin Kosmosda Computer Vision

Bu notebook dərin kosmik obyektləri (dumanlıqlar, qalaktikalar, ulduz topaları)
**YOLOv8** modeli ilə aşkarlayır.

## 📋 Addımlar:
1. Mühitin qurulması
2. Datasetin yüklənməsi
3. Datasetin araşdırılması (EDA)
4. Modelin öyrədilməsi
5. Modelin qiymətləndirilməsi
6. Şəkil/Video analizi

> ⚠️ **Runtime → Change runtime type → GPU (T4)** seçin!

---
## 1️⃣ Mühitin Qurulması
Lazımi kitabxanaları yükləyirik və layihəni GitHub-dan klonlayırıq.

In [ ]:
# GPU yoxla — T4 və ya daha yuxarı olmalıdır
!nvidia-smi

import torch
print(f"\n✅ PyTorch: {torch.__version__}")
print(f"🖥️  CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Layihəni GitHub-dan klonla
!git clone https://github.com/YOUR_USERNAME/DeepSpaceVision.git
%cd DeepSpaceVision

# Bağımlılıqları yüklə
!pip install -r requirements.txt -q

In [ ]:
# Əsas kitabxanaları import et
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO
from IPython.display import Image, display, HTML

# Layihə funksiyalarını import et
from utils.dataset import split_dataset, verify_dataset, create_sample_labels
from utils.visualize import plot_detection_results, plot_training_metrics, plot_class_distribution
from utils.augment import augment_dataset

print("✅ Bütün kitabxanalar uğurla yükləndi!")

---
## 2️⃣ Datasetin Yüklənməsi

Dərin kosmik obyektlərin şəkillərini yükləyirik.

### Seçim A: Roboflow-dan hazır dataset
### Seçim B: Öz şəkillərinizi yükləyin

In [ ]:
# ============================================
# SEÇİM A: Roboflow-dan DeepSpace dataset
# ============================================
# Roboflow-dan hazır, annotasiya edilmiş dataset yükləyirik
# Bu, ən asan yoldur!

!pip install roboflow -q

from roboflow import Roboflow

# Roboflow API açarınızı bura yazın
# https://app.roboflow.com/ saytından əldə edə bilərsiniz (pulsuz)
ROBOFLOW_API_KEY = "YOUR_API_KEY"  # <-- Bunu dəyişdirin!

rf = Roboflow(api_key=ROBOFLOW_API_KEY)

# Deep Space Objects dataseti
# Alternativ olaraq öz datasetinizi Roboflow-da yarada bilərsiniz
project = rf.workspace().project("deep-space-objects")
dataset = project.version(1).download("yolov8")

print(f"\n✅ Dataset yükləndi: {dataset.location}")

In [ ]:
# ============================================
# SEÇİM B: Öz şəkillərinizi yükləyin
# ============================================
# Google Drive-dan və ya yerli fayllardan

# Google Drive-ı bağla
from google.colab import drive
drive.mount('/content/drive')

# Əgər şəkilləriniz Drive-dadırsa:
# !cp -r /content/drive/MyDrive/space_images/* data/images/
# !cp -r /content/drive/MyDrive/space_labels/* data/labels/

# Əgər Colab-a fayl yükləmək istəyirsinizsə:
from google.colab import files
# uploaded = files.upload()  # Uncomment edib faylları yükləyin

---
## 3️⃣ Datasetin Araşdırılması (EDA)
Şəkilləri və etiketləri vizual olaraq yoxlayırıq.

In [ ]:
# Dataset strukturunu yoxla
verify_dataset('data')

In [ ]:
# Sinif paylanmasını göstər
class_names = {0: 'nebula', 1: 'galaxy', 2: 'star_cluster'}
plot_class_distribution('data/labels/train', class_names)

In [ ]:
# Nümunə şəkilləri göstər
train_images_dir = Path('data/images/train')
if train_images_dir.exists():
    images = list(train_images_dir.glob('*'))[:9]
    
    fig, axes = plt.subplots(3, 3, figsize=(15, 15))
    fig.suptitle('📸 Nümunə Kosmik Şəkillər', fontsize=18, fontweight='bold')
    
    for idx, ax in enumerate(axes.flatten()):
        if idx < len(images):
            img = cv2.imread(str(images[idx]))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            ax.imshow(img)
            ax.set_title(images[idx].name, fontsize=10)
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()
else:
    print('⚠️ Əvvəlcə dataseti yükləyin (Addım 2)')

---
## 4️⃣ Model Öyrətmə (Training)

YOLOv8 modelini öz datasetimiz üzərində öyrədirik.

**Parametrlər:**
- `model`: YOLOv8 ölçüsü (n/s/m/l/x) — n ən kiçik, x ən böyük
- `epochs`: Neçə dəfə bütün datasetin üzərindən keçəcək
- `batch`: Hər addımda neçə şəkil istifadə edəcək
- `imgsz`: Şəkil ölçüsü (640 standartdır)

In [ ]:
# ========================================
# MODEL ÖYRƏTMƏ
# ========================================

# YOLOv8 modelini yüklə (pretrained — ImageNet üzərində öncədən öyrədilmiş)
# Ölçülər: yolov8n (ən kiçik/sürətli) → yolov8x (ən böyük/dəqiq)
model = YOLO('yolov8n.pt')  # nano model — Colab T4 üçün ideal

# Öyrətməni başlat
results = model.train(
    data='configs/deep_space.yaml',  # Dataset konfiqurasiyası
    epochs=50,                        # Epoch sayı (50-100 arası ideal)
    batch=16,                         # Batch ölçüsü (GPU yaddaşına görə)
    imgsz=640,                        # Şəkil ölçüsü
    device='cuda',                    # GPU istifadə et
    workers=2,                        # Data yükləyicilər
    patience=15,                      # Early stopping
    save=True,                        # Ən yaxşı modeli saxla
    plots=True,                       # Qrafiklər yarat
    cos_lr=True,                      # Cosine learning rate
    flipud=0.5,                       # Şaquli əksetdirmə (kosmos üçün faydalı)
    mosaic=1.0,                       # Mozaik augmentasiya
    verbose=True,                     # Ətraflı çıxış
)

print("\n✅ Öyrətmə tamamlandı!")

In [ ]:
# Öyrətmə nəticələrini göstər
# YOLO avtomatik qrafiklər yaradır:

from IPython.display import Image as IPImage, display
import glob

# Son öyrətmə qovluğunu tap
run_dirs = sorted(glob.glob('runs/detect/*/'), key=os.path.getmtime)
if run_dirs:
    last_run = run_dirs[-1]
    print(f"📁 Son öyrətmə: {last_run}")
    
    # Nəticə qrafiklərini göstər
    for plot_name in ['results.png', 'confusion_matrix.png', 
                       'F1_curve.png', 'P_curve.png', 'R_curve.png']:
        plot_path = os.path.join(last_run, plot_name)
        if os.path.exists(plot_path):
            print(f"\n📊 {plot_name}:")
            display(IPImage(filename=plot_path, width=800))

---
## 5️⃣ Model Qiymətləndirmə
Öyrədilmiş modeli validation/test dataseti üzərində qiymətləndiririk.

In [ ]:
# Ən yaxşı modeli yüklə
best_model_path = os.path.join(last_run, 'weights', 'best.pt')
best_model = YOLO(best_model_path)

# Validation dataseti üzərində qiymətləndir
metrics = best_model.val(
    data='configs/deep_space.yaml',
    split='val',
    plots=True,
)

print(f"\n📊 Qiymətləndirmə Nəticələri:")
print(f"   mAP@0.5:      {metrics.box.map50:.4f}")
print(f"   mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"   Precision:     {metrics.box.mp:.4f}")
print(f"   Recall:        {metrics.box.mr:.4f}")

---
## 6️⃣ Şəkil və Video Analizi (İnferens)
Öyrədilmiş modeli yeni şəkillər və videolar üzərində sınayaq!

In [ ]:
# ========================================
# ŞƏKİL ANALİZİ
# ========================================

# Test şəklini yüklə (Colab-dan)
from google.colab import files
print("📤 Kosmik şəkil yükləyin:")
uploaded = files.upload()

# Yüklənən şəkli analiz et
for filename in uploaded.keys():
    print(f"\n🔍 Analiz edilir: {filename}")
    
    # Model ilə aşkarlama
    results = best_model(filename, conf=0.25)
    
    # Nəticəni göstər
    annotated = results[0].plot()  # Annotasiya edilmiş şəkil
    annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
    
    plt.figure(figsize=(14, 10))
    plt.imshow(annotated_rgb)
    plt.title(f'🔭 DeepSpaceVision — {filename}', fontsize=16, fontweight='bold')
    plt.axis('off')
    plt.show()
    
    # Tapılan obyektləri göstər
    boxes = results[0].boxes
    if boxes is not None and len(boxes) > 0:
        print(f"\n🎯 Aşkarlanan obyektlər:")
        for box in boxes:
            cls_id = int(box.cls[0])
            cls_name = results[0].names[cls_id]
            conf = float(box.conf[0])
            print(f"   • {cls_name}: {conf:.2%}")
    else:
        print("   Heç bir obyekt aşkarlanmadı. Etibar həddini azaldın (conf=0.15).")

In [ ]:
# ========================================
# VİDEO ANALİZİ
# ========================================

# Video yüklə
print("📤 Kosmik video yükləyin (.mp4):")
uploaded_video = files.upload()

for filename in uploaded_video.keys():
    print(f"\n🎥 Video analiz edilir: {filename}")
    
    # detect.py istifadə et
    from detect import detect_video, load_model
    
    result_path, summary = detect_video(
        model=best_model,
        source=filename,
        conf_threshold=0.25,
        save_dir='results'
    )
    
    # Nəticə videosunu göstər
    from IPython.display import Video
    if result_path:
        display(Video(result_path, width=800))

In [ ]:
# ========================================
# URL-DƏN ŞƏKİL ANALİZİ
# ========================================
# İnternetdən birbaşa kosmik şəkil analiz edin

import urllib.request

# NASA / Hubble şəkillər (nümunə URL-lər)
test_urls = [
    # Orion Dumanlığı
    'https://upload.wikimedia.org/wikipedia/commons/thumb/f/f3/Orion_Nebula_-_Hubble_2006_mosaic_18000.jpg/1280px-Orion_Nebula_-_Hubble_2006_mosaic_18000.jpg',
    # Andromeda Qalaktikası
    'https://upload.wikimedia.org/wikipedia/commons/thumb/9/98/Andromeda_Galaxy_%28with_h-alpha%29.jpg/1280px-Andromeda_Galaxy_%28with_h-alpha%29.jpg',
]

for i, url in enumerate(test_urls):
    filename = f'test_image_{i}.jpg'
    print(f"\n⬇️  Yüklənir: {url[:60]}...")
    urllib.request.urlretrieve(url, filename)
    
    # Analiz et
    results = best_model(filename, conf=0.20)
    annotated = results[0].plot()
    annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
    
    plt.figure(figsize=(14, 10))
    plt.imshow(annotated_rgb)
    plt.title(f'🔭 Test {i+1}', fontsize=16, fontweight='bold')
    plt.axis('off')
    plt.show()

---
## 7️⃣ Modeli İxrac Et və Saxla
Öyrədilmiş modeli saxlayıb, Google Drive-a yükləyin.

In [ ]:
# Modeli ONNX formatına ixrac et (universal format)
best_model.export(format='onnx')
print("\n✅ Model ONNX formatına ixrac edildi!")

# Google Drive-a kopyala
import shutil
drive_path = '/content/drive/MyDrive/DeepSpaceVision_Models/'
os.makedirs(drive_path, exist_ok=True)

shutil.copy2(best_model_path, drive_path + 'best.pt')
print(f"💾 Model Drive-a kopyalandı: {drive_path}")

# Və ya birbaşa yüklə
# files.download(best_model_path)

---
## 🎉 Tamamlandı!

**Növbəti addımlar:**
- Daha çox data ilə modeli yenidən öyrədin
- `yolov8s.pt` və ya `yolov8m.pt` ilə daha dəqiq model əldə edin
- Real teleskop videoları ilə sınayın
- Web tətbiqi yaradıb modeli deploy edin

**📧 Suallarınız varsa issue açın!**